<a href="https://colab.research.google.com/github/S-Ananth7/Genai_agent_foundation/blob/main/ai_agentic_frameworks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langgraph langchain-openapi langchain_community python-dotenv ddgs

In [ ]:
pip install -q langchain_openai langchain-google-genai google-search-results

In [ ]:
# from dotenv import load_dotenv
# import os

In [ ]:
from google.colab import userdata
# userdata.get('secretName')

In [ ]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

# Load API keys from Colab Secrets
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
SERP_API_KEY = userdata.get("SERP_API_KEY")

if not GEMINI_API_KEY:
    GEMINI_API_KEY = input("Enter your GEMINI_API_KEY: ").strip()

if not SERP_API_KEY:
    SERP_API_KEY = input("Enter your SERP_API_KEY: ").strip()

print("API keys loaded successfully!")

# Create Gemini model
model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    google_api_key=GEMINI_API_KEY
)

# Test
response = model.invoke("What are AI agents?")
print(response.content)

In [ ]:
# from langchain_openai import ChatOpenAI

# model = ChatOpenAI(model="gpt-5-mini", temperature=0, max_tokens=None,timeout=None,max_retries=2, api_key=OPENAI_API_KEY)
# response = model.invoke("Whar are AI agents?")
# print(response.content)



In [ ]:
import numexpr
import math
from typing import Dict, Any

from langchain_core.tools import tool
from langchain_community.utilities import SerpAPIWrapper


@tool("internet_search")
def internet_search(query: str) -> str:
    """Search Google via SerpAPI for up to date information."""
    params ={"engine": "google", "gl":"us", "hl":"en"}
    search = SerpAPIWrapper( params=params, serpapi_api_key=SERP_API_KEY)
    return search.run(query)

@tool("calculator")
def calculator(expression: str) -> str:
    """Evaluate a single line mathematical expression with numexpr."""
    local_dict= {"pi": math.pi, "e": math.e}
    out = numexpr.evaluate(
        expression.strip(),
        local_dict=local_dict,
        global_dict={}
    )
    return str(out)

tools = [internet_search, calculator]

tool_map: Dict[str, Any] = {t.name: t for t in tools}


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0,
    google_api_key=GEMINI_API_KEY
).bind_tools(
    tools,
    tool_choice="any"
)

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage

def run_once(prompt: str, max_steps: int = 4) -> str:
  messages = [HumanMessage(content=prompt)]
  for _ in range(max_steps):
    ai: AIMessage = llm.invoke(messages)
    messages.append(ai)

    calls = getattr(ai, "tool_calls", None) or []
    if not calls:
      return ai.content

    for call in calls:
      name = call["name"]
      args = call.get("args",{})
      result = tool_map[name].invoke(args)
      messages.append(ToolMessage(
          content = str(result),
          name=name,
          tool_call_id=call["id"]
      ))

    return messages[-1].content

print(run_once("""Two step task.


Step 1: Use internet_search to get the current air temperature in New York City today. Show the exact query you used, the top source title and snippet, and extract a numeric temperature in Celsius. Return this temperature as feedback for Step 2.

Step 2: Using the Celsius value from Step 1, compute its square with calculator. Show the exact expression you used and the numeric result.

Important: Give a short final answer in this format:
Current temperature:
Square of current temperature:"""))



In [ ]:
from typing import List, Dict , Any, TypedDict, Annotated


def run_once(prompt: str, max_steps: int = 8) -> str:
    messages: List[HumanMessage | AIMessage | ToolMessage] = [
        HumanMessage(content=prompt)
    ]

    for _ in range(max_steps):
        ai: AIMessage = llm.invoke(messages)
        messages.append(ai)

        calls = getattr(ai, "tool_calls", None) or []
        if not calls:

            break
        for call in calls:
            name = call["name"]
            args = call.get("args", {}) or {}
            result = tool_map[name].invoke(args)

            messages.append(
                ToolMessage(
                    content=str(result),
                    name=name,
                    tool_call_id=call.get("id"),
                )
            )

    messages.append(
        HumanMessage(
            content=(
                "Finish now. Give a short final answer in this exact format:\n\n"
                "Current temperature:\nSquare of current temperature:"
            ).strip()
        )
    )

    final_ai: AIMessage = llm.invoke(messages)
    return final_ai.content


print(run_once("""Two step task.

Step 1: Use internet_search to get the current air temperature in New York City today. Show the exact query you used, the top source title and snippet, and extract a numeric temperature in Celsius. Return this temperature as feedback for Step 2.

Step 2: Using the Celsius value from Step 1, compute its square with calculator. Show the exact expression you used and the numeric result.

Important: Give a short final answer in this format:
Current temperature:
Square of current temperature:"""

))

In [ ]:
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

class AgentState(TypeDict):
    messages: Annotated[List[BasMessage], add_messages]

llm = ChatOpenAPI(model="gpt-4o", temperature=0, max_tokens=800).bind_tools(tools)

def llm_node(state: AgentState) -> AgentState:
    ai = llm.invoke(state["messages"])
    return {"messages" : [ai]}

tool_node = ToolNode(tools=tools)

graph = StateGraph(AgentState)
graph.add_node("llm", llm_node)
graph.add_node("tools", tool_node)
graph.add_edge(START, "llm")

def route(state: AgentState):
    last = state["messages"][-1]
    calls = getattr(last, "tool_calls", None) or []
    return "tools" if calls else END

graph.add_conditional_edges("llm", route, {"tools": "tools", END: END})
graph.add_edge("tools", "llm")

Compile with in-memory checkpoints and configure a thread

In [ ]:
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

Trace execution and inspect state and memory

In [ ]:
import json

def short(msg: BaseMessage, max_len: int =140 ) -> str:
  """Compact one-line view of a message."""
  role = type(msg).__name__.replace("Message", "").lower()
  content=getattr(msg, "content", "")
  if isinstance(content, list):
    # some tool outputs can be list payloads
    try:
      content = json.dumps(content)
    except Exception:
      content = str(content)
  text = str(content).replace("\n", " ").strip()
  if len(text) > max_len:
    text = text[: max_len-3] + "..."
  # include tool name or function call info when available
  if hasattr(msg, "tool_calls") and getattr(msg, "tool_calls"):
    tnames=[tc.get("name","tool") for tc in msg.tool_calls]
    return f"{role}: tool_calls -> {tnames}"
  if isinstance(msg, ToolMessage):
    return f"{role}{msg.name}): {text}"
  return f"{role}: {text}"



Trace execution and inspect state and memory

In [ ]:
def print_state_snapshot(app, config, title: str):
  "Print current graph state and memory for a given thread."
  snap = app.get_state(config)
  values = snap.values or {}
  msgs: List[BaseMessage] = values.get("messages", [])
  print(f"\n==={title} | state snapshot ===")
  print(f"messages: {len(msgs)} total")
  for i, m in enumerate(msgs[-5:], start=max(0, len(msgs)-5) + 1):
    print(f" {i:>3}: {_short(m)}")
  # show routing info and queued tasks if present
  nxt = getattr(snap, "next", None)
  tasks = getattr(snap, "tasks", None)
  if nxt:
    print(f"next nodes: {list(nxt)}")
  if tasks:
    print(f"queued tasks: {tasks}")

  print("memory: in-memory checkpoint present for this thread")


In [ ]:
def run_with_tracing(app, input_state: AgentState, config, title: str):
  """Run the graph while printing per-node updates and final memory."""
  print(f"\n==={title} | execution trace ===")
  final = None
  # stream_mode="updates" surfaces node-level updates
  for event in app.stream(input_state, config=config, stream_mode="updates"):
    for node, upd in event.items():
      keys = list(upd.keys())
      print(f"[enter {node}] updated: {keys}")
      msgs = upd.get("messages") or []
      if msgs:
        print(f" {_short(msgs[-1])}")
      print(f"[Leave {next}")
      final = upd

  print_state_snapshot(app, config, title=f"{title} | after run")
  snap = app.get_state(config)
  msgs = snap.values.get("messages", [])
  return msgs[-1].content if msgs else ""

In [ ]:
cfg= {"configurable": {"thread_id": "nyc-weather-session"}}

Turn 1: get the current NYC air temperature in Celsius

In [ ]:
turn1_answer = run_with_tracing(
    app, {"messages": [HumanMessage(content="Get the current air temperature in New York City in Celsius")]},
    config={**cfg, "recursion_limit": 20},
    title="TURN 1",
)
print("\nTURN 1 (final assistant):\n", turn1_answer)

Turn 2: square the temperature in the same thread

In [ ]:
turn2_answer = run_with_tracing(
    app, {"messages": [HumanMessage(content="Now compute the square of the temperature.")]}
    config={**cfg, "recursion_limit":20}
    title="TURN 2"
)
print("\nTURN 2 (final assistnat):\n", turn2_answer)

Branch a parallel thread for a different follow-up

In [ ]:
cfg_branch = {"configurable" : {"thread_id": "nyc-weather-session-branch"}}
branch_answer = run_with_tracing(
    app, {"messages":[HumanMessage(content="Instead of squaring, convert to to Fahrenheit and report both.")]},
    config=cfg_branch,
    title="TURN 2",
)

print("\nBRANCH (final assistant):\n", branch_answer)


In [ ]:
print_state_snapshot(app, cfg, title="MAIN THREAD memory view")
print_state_snapshot(app, cfg_branch, title="BRANCH THREAD memory view")